# 10. Experiment 1 — Word-context representation

In [2]:
!pip install nltk

In [5]:
import numpy as np
import nltk
nltk.download('punkt_tab')
from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity

toy_corpus = [
    "the doctor treated the patient in the hospital",
    "the physician treated the patient in the clinic",
    "the cat eats fish and drinks milk",
    "the dog eats meat and likes bones",
    "the doctor drives a car",
    "the physician drives a car",
    "the monkey eats a banana"
]

def build_word_context_matrix(corpus, window):
    tokenized_corpus = [nltk.word_tokenize(doc.lower()) for doc in corpus]
    vocab = set()
    for doc in tokenized_corpus:
        vocab.update(doc)

    vocab = sorted(list(vocab))
    word2idx = {w: i for i, w in enumerate(vocab)}
    V = len(vocab)

    matrix = np.zeros((V, V))

    for doc in tokenized_corpus:
        doc_len = len(doc)
        for i, target_word in enumerate(doc):
            target_idx = word2idx[target_word]

            start = max(0, i - window)
            end = min(doc_len, i + window + 1)

            for j in range(start, end):
                if i != j:
                    context_word = doc[j]
                    context_idx = word2idx[context_word]
                    matrix[target_idx, context_idx] += 1

    return matrix, word2idx, vocab

def analyze_matrix(matrix, word2idx, window):
    V = len(word2idx)
    matrix_size = V * V
    non_zeros = np.count_nonzero(matrix)

    doc_vec = matrix[word2idx['doctor']].reshape(1, -1)
    phy_vec = matrix[word2idx['physician']].reshape(1, -1)
    cat_vec = matrix[word2idx['cat']].reshape(1, -1)

    sim_doc_phy = cosine_similarity(doc_vec, phy_vec)[0][0] if np.sum(doc_vec) and np.sum(phy_vec) else 0.0
    sim_doc_cat = cosine_similarity(doc_vec, cat_vec)[0][0] if np.sum(doc_vec) and np.sum(cat_vec) else 0.0

    print(f"--- WINDOW = {window} ---")
    print(f"Vocabulary size:     {V}")
    print(f"Matrix size:         {V} x {V} = {matrix_size} elements")
    print(f"Non-zero values:     {non_zeros} (Sparsity: {100 - (non_zeros/matrix_size*100):.2f}%)")
    print(f"Cos(doctor, physician): {sim_doc_phy:.4f}")
    print(f"Cos(doctor, cat):       {sim_doc_cat:.4f}\n")

for w in [1, 2, 5]:
    mat, w2i, vocab = build_word_context_matrix(toy_corpus, window=w)
    analyze_matrix(mat, w2i, w)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


--- WINDOW = 1 ---
Vocabulary size:     23
Matrix size:         23 x 23 = 529 elements
Non-zero values:     60 (Sparsity: 88.66%)
Cos(doctor, physician): 1.0000
Cos(doctor, cat):       0.5774

--- WINDOW = 2 ---
Vocabulary size:     23
Matrix size:         23 x 23 = 529 elements
Non-zero values:     94 (Sparsity: 82.23%)
Cos(doctor, physician): 1.0000
Cos(doctor, cat):       0.5000

--- WINDOW = 5 ---
Vocabulary size:     23
Matrix size:         23 x 23 = 529 elements
Non-zero values:     159 (Sparsity: 69.94%)
Cos(doctor, physician): 1.0000
Cos(doctor, cat):       0.3482



# 11. Core implementation

In [6]:
import numpy as np
import nltk

def build_vocabulary(corpus):
    """Xây dựng từ điển ánh xạ từ -> index và index -> từ"""
    vocab = set()
    for doc in corpus:
        tokens = nltk.word_tokenize(doc.lower())
        vocab.update(tokens)

    vocab = sorted(list(vocab))
    word2idx = {w: i for i, w in enumerate(vocab)}
    idx2word = {i: w for i, w in enumerate(vocab)}

    return word2idx, idx2word, len(vocab)

def build_cooccurrence_matrix(corpus, word2idx, window=2):
    """Xây dựng ma trận co-occurrence dựa trên context window"""
    V = len(word2idx)
    matrix = np.zeros((V, V))

    for doc in corpus:
        tokens = nltk.word_tokenize(doc.lower())
        length = len(tokens)

        for i, target_word in enumerate(tokens):
            if target_word not in word2idx:
                continue
            target_idx = word2idx[target_word]

            start = max(0, i - window)
            end = min(length, i + window + 1)

            for j in range(start, end):
                if i != j:
                    context_word = tokens[j]
                    if context_word in word2idx:
                        context_idx = word2idx[context_word]
                        matrix[target_idx, context_idx] += 1

    return matrix

def cosine_similarity(vec_a, vec_b):
    """Tính Cosine Similarity giữa 2 vector bằng công thức toán học"""
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot_product / (norm_a * norm_b)

def most_similar(word, matrix, word2idx, idx2word, top_k=5):
    if word not in word2idx:
        return f"Từ '{word}' không có trong từ điển."

    target_idx = word2idx[word]
    target_vec = matrix[target_idx]

    similarities = []
    for i in range(len(matrix)):
        if i != target_idx:
            sim = cosine_similarity(target_vec, matrix[i])
            similarities.append((idx2word[i], sim))

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]


In [7]:
toy_corpus = [
    "the doctor treated the patient in the hospital",
    "the physician treated the patient in the clinic",
    "the cat eats fish and drinks milk",
    "the dog eats meat and likes bones",
    "the doctor drives a car",
    "the physician drives a car"
]

w2i, i2w, vocab_size = build_vocabulary(toy_corpus)
print(f"Vocabulary Size: {vocab_size}")

X = build_cooccurrence_matrix(toy_corpus, w2i, window=2)

print("\nTop 5 từ gần nghĩa nhất với 'doctor':")
results = most_similar(word="doctor", matrix=X, word2idx=w2i, idx2word=i2w, top_k=5)
for w, sim in results:
    print(f" - {w:<10}: {sim:.4f}")

Vocabulary Size: 21

Top 5 từ gần nghĩa nhất với 'doctor':
 - physician : 1.0000
 - patient   : 0.8250
 - in        : 0.7385
 - treated   : 0.7385
 - drives    : 0.6172


# 17. Experiment 2 — Word2Vec

In [8]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 44.0 MB/s eta 0:00:00


In [9]:
import nltk
from gensim.models import Word2Vec

nltk.download('punkt')
nltk.download('punkt_tab')

raw_corpus = [
    "the doctor treated the patient in the hospital",
    "the physician treated the patient in the clinic",
    "the cat eats fish and drinks milk",
    "the dog eats meat and likes bones",
    "the doctor drives a car",
    "the physician drives a car",
    "the monkey eats a banana",
    "the nurse helps the doctor in the hospital"
]

tokenized_corpus = [nltk.word_tokenize(doc.lower()) for doc in raw_corpus]

w2v_model = Word2Vec(
    sentences=tokenized_corpus,
    vector_size=50,
    window=2,
    min_count=1,
    epochs=20,
    sg=0
)

vocab_size = len(w2v_model.wv)
print("Huấn luyện hoàn tất!\n")
print(f"--- THÔNG SỐ HUẤN LUYỆN WORD2VEC ---")
print(f"Corpus size:         {len(tokenized_corpus)} documents")
print(f"Vocabulary size:     {vocab_size} words")
print(f"Embedding dimension: {w2v_model.vector_size}")
print(f"Context window:      {w2v_model.window}")
print(f"Min count:           {w2v_model.min_count}")
print(f"Epochs:              {w2v_model.epochs}")
print(f"Training objective:  {'CBOW' if w2v_model.sg == 0 else 'Skip-gram'}")
print("-" * 35)

word = "doctor"
if word in w2v_model.wv:
    print(f"\nTop 3 từ giống '{word}' nhất theo Word2Vec:")
    for w, sim in w2v_model.wv.most_similar(word, topn=3):
        print(f" - {w:<10}: {sim:.4f}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Đang huấn luyện Word2Vec model...
Huấn luyện hoàn tất!

--- THÔNG SỐ HUẤN LUYỆN WORD2VEC ---
Corpus size:         8 documents
Vocabulary size:     25 words
Embedding dimension: 50
Context window:      2
Min count:           1
Epochs:              20
Training objective:  CBOW
-----------------------------------

Top 3 từ giống 'doctor' nhất theo Word2Vec:
 - eats      : 0.1678
 - nurse     : 0.1566
 - fish      : 0.1410


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# 18. Inspect embeddings

In [14]:
print("Đang kết nối với Google Drive...")
drive.mount('/content/drive')


Đang kết nối với Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import gzip
import json
import nltk
from google.colab import drive
from gensim.models import Word2Vec

nltk.download('punkt')

file_path = '/content/drive/MyDrive/Colab_Notebooks/c4-train.00000-of-01024-30K.json.gz'

real_corpus = []
with gzip.open(file_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        if 'text' in data:
            real_corpus.append(data['text'])

print(f"Đã tải xong {len(real_corpus)} văn bản!")

tokenized_real_corpus = [nltk.word_tokenize(doc.lower()) for doc in real_corpus]

print("\nĐang huấn luyện Word2Vec...")
real_model = Word2Vec(
    sentences=tokenized_real_corpus,
    vector_size=100,
    window=5,
    min_count=5,
    epochs=5
)

print("\nHuấn luyện hoàn tất!\n")
print("-" * 40)

words_to_check = ["doctor", "hospital", "patient", "disease", "computer", "football", "banana"]

for word in words_to_check:
    if word in real_model.wv:
        print(f"\nTop-5 từ gần '{word}':")
        for w, sim in real_model.wv.most_similar(word, topn=5):
            print(f" - {w:<12}: {sim:.4f}")
    else:
        print(f"\nTừ '{word}' không có trong từ vựng (do xuất hiện < 5 lần)!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Đã tải xong 30000 văn bản!

Đang huấn luyện Word2Vec...

Huấn luyện hoàn tất!

----------------------------------------

Top-5 từ gần 'doctor':
 - dentist     : 0.8425
 - physician   : 0.7956
 - veterinarian: 0.7723
 - therapist   : 0.7244
 - child       : 0.7081

Top-5 từ gần 'hospital':
 - clinic      : 0.8241
 - clinics     : 0.6484
 - cincinnati  : 0.6436
 - daycare     : 0.6336
 - nursing     : 0.6220

Top-5 từ gần 'patient':
 - physician   : 0.6962
 - diagnosis   : 0.6752
 - patients    : 0.6695
 - therapy     : 0.6668
 - child       : 0.6629

Top-5 từ gần 'disease':
 - chronic     : 0.8548
 - infection   : 0.8321
 - diseases    : 0.8157
 - obesity     : 0.8080
 - arthritis   : 0.7937

Top-5 từ gần 'computer':
 - desktop     : 0.7972
 - device      : 0.7812
 - laptop      : 0.7803
 - browser     : 0.7522
 - router      : 0.7285

Top-5 từ gần 'football':
 - baseball    : 0.8061
 - basketball  : 0.7415
 - soccer      : 0.7407
 - league      : 0.7142
 - hockey      : 0.7106

Top-5 t

**Tại sao những từ trên (như physician, nurse, patient, hospital) lại gần "doctor", trong khi computer, football, banana thì không?**

- Thuật toán Word2Vec cập nhật vector dựa trên việc các từ chia sẻ chung một context window. Trong hàng chục nghìn văn bản của tập corpus, từ doctor liên tục xuất hiện sát cạnh hoặc chia sẻ chung các từ lân cận như: "treated", "examined", "prescribed", "clinic", "health".

- Quan hệ đồng xuất hiện (Co-occurrence): Các từ hospital, patient, disease không thể thay thế trực tiếp cho doctor, nhưng chúng lại trực tiếp nằm bên trong cửa sổ ngữ cảnh của doctor với tần suất cực kỳ dày đặc.

- Sự xa cách của các từ không liên quan: Ngược lại, computer, football, banana hầu như không bao giờ chia sẻ các context words như "treated" hay "clinic". Cửa sổ ngữ cảnh của chúng bao quanh bởi "software", "stadium", "peel". zero co-occurrence trong các văn bản y tế khiến vector của chúng bị đẩy ra xa, dẫn đến điểm similarity cực thấp hoặc âm.

# 19. Experiment 3 — Context window

In [17]:
from gensim.models import Word2Vec

results = {
    "doctor-physician": [],
    "doctor-hospital": [],
    "cat-dog": []
}

windows = [2, 5, 10]

print("Đang huấn luyện 3 mô hình với window = 2, 5, 10...\n")
for w in windows:
    model = Word2Vec(
        sentences=tokenized_real_corpus,
        vector_size=100,
        window=w,
        min_count=5,
        epochs=5
    )

    for pair in results.keys():
        w1, w2 = pair.split("-")
        if w1 in model.wv and w2 in model.wv:
            sim = model.wv.similarity(w1, w2)
            results[pair].append(f"{sim:.4f}")
        else:
            results[pair].append("N/A")

print(f"| Word pair | Window 2 | Window 5 | Window 10 |")
print(f"| :--- | :--- | :--- | :--- |")
for pair, scores in results.items():
    print(f"| {pair} | {scores[0]} | {scores[1]} | {scores[2]} |")

Đang huấn luyện 3 mô hình với window = 2, 5, 10...

| Word pair | Window 2 | Window 5 | Window 10 |
| :--- | :--- | :--- | :--- |
| doctor-physician | 0.8010 | 0.8156 | 0.7984 |
| doctor-hospital | 0.4958 | 0.5128 | 0.5338 |
| cat-dog | 0.7633 | 0.7840 | 0.7924 |


**Window lớn hơn có luôn tốt hơn không?**

Trả lời: KHÔNG. Không có kích thước cửa sổ (window size) nào là tốt nhất tuyệt đối, mà nó phụ thuộc hoàn toàn vào mục đích của bài toán xử lý ngôn ngữ tự nhiên:
- Window nhỏ ($w = 1, 2$): Giúp nắm bắt Cú pháp và Từ đồng nghĩa. Ở kích thước nhỏ, thuật toán chỉ quan tâm đến các từ sát vách. Nó rất nhạy bén trong việc tìm ra các từ có cùng từ loại hoặc có thể thay thế trực tiếp cho nhau trong câu. Vì vậy, cặp doctor–physician hoặc cat–dog thường sẽ có điểm similarity rất cao ở window 2.
- Window lớn ($w = 5, 10$): Nắm bắt Ngữ nghĩa chủ đề. Cửa sổ rộng cho phép mô hình nhìn bao quát cả câu hoặc cả đoạn văn. Nó bắt đầu liên kết các từ có liên quan về mặt nội dung, không cần phải đứng gần nhau hoặc cùng từ loại. Do đó, điểm similarity của cặp doctor–hospital sẽ tăng mạnh khi window mở rộng ra 5 hoặc 10.

# 20. Experiment 4 — Embedding dimension

In [19]:
import time
import os
from gensim.models import Word2Vec

dims = [50, 100, 300]
results_dim = []

for d in dims:
    start_time = time.time()

    model = Word2Vec(sentences=tokenized_real_corpus, vector_size=d, window=5, min_count=5, epochs=5)

    train_time = time.time() - start_time

    model_path = f"model_{d}.wv"
    model.save(model_path)
    model_size_mb = os.path.getsize(model_path) / (1024 * 1024)

    sim = model.wv.similarity("doctor", "hospital") if "doctor" in model.wv and "hospital" in model.wv else 0.0

    results_dim.append((d, train_time, model_size_mb, sim))

print("| Dimension | Training Time (s) | Model Size (MB) | Cos(doctor, hospital) |")
print("| :--- | :--- | :--- | :--- |")
for r in results_dim:
    print(f"| {r[0]} | {r[1]:.2f} | {r[2]:.2f} | {r[3]:.4f} |")

| Dimension | Training Time (s) | Model Size (MB) | Cos(doctor, hospital) |
| :--- | :--- | :--- | :--- |
| 50 | 134.24 | 22.86 | 0.5664 |
| 100 | 136.05 | 44.11 | 0.4971 |
| 300 | 164.22 | 1.61 | 0.4784 |


# 21. Evaluation — Word similarity

In [20]:
word_pairs = [
    ("doctor", "physician"),
    ("car", "automobile"),
    ("king", "queen"),
    ("cat", "dog"),
    ("computer", "banana")
]

print("| Word Pair | Similarity Score | Ghi chú |")
print("| :--- | :--- | :--- |")

results = []
for w1, w2 in word_pairs:
    if w1 in real_model.wv and w2 in real_model.wv:
        sim = real_model.wv.similarity(w1, w2)
        results.append(((w1, w2), sim))
    else:
        results.append(((w1, w2), -999))

results.sort(key=lambda x: x[1], reverse=True)

for (w1, w2), sim in results:
    if sim == -999:
        print(f"| {w1} - {w2} | N/A (Out of vocab) | Thiếu dữ liệu |")
    else:
        print(f"| {w1} - {w2} | {sim:.4f} | |")

| Word Pair | Similarity Score | Ghi chú |
| :--- | :--- | :--- |
| king - queen | 0.8142 | |
| doctor - physician | 0.7956 | |
| cat - dog | 0.7813 | |
| car - automobile | 0.5955 | |
| computer - banana | -0.1016 | |


# 22. Evaluation — Word analogy

In [21]:
try:
    result = real_model.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=1)
    print(f"king - man + woman ≈ {result[0][0]} (Score: {result[0][1]:.4f})")
except KeyError as e:
    print(f"Thiếu từ vựng trong corpus: {e}")

king - man + woman ≈ queen (Score: 0.7866)


**Tại sao vector arithmetic có thể biểu hiện các quan hệ này?**

Phép toán $v_{\text{king}} - v_{\text{man}} + v_{\text{woman}} \approx v_{\text{queen}}$ hoạt động dựa trên các khoảng cách và hướng (spatial offsets) sinh ra từ xác suất đồng xuất hiện trong không gian hình học đa chiều, cụ thể:Vector hóa đặc trưng:
- Phép trừ $v_{\text{king}} - v_{\text{man}}$ triệt tiêu các đặc trưng chung về "con người/nam giới", để lại một hướng vector cốt lõi mang ý nghĩa "sự cai trị" hoặc "hoàng gia".
- Phép tịnh tiến (Translation): Khi cộng hướng vector "hoàng gia" này vào $v_{\text{woman}}$, ta tịnh tiến tọa độ của từ "woman" đến một vùng không gian mới. Vùng này đại diện cho sự giao thoa ngữ cảnh giữa "phụ nữ" và "hoàng gia".
- Hội tụ ngữ cảnh: Nhờ quá trình tối ưu hóa của mạng nơ-ron, từ "queen" đã được đẩy về chính tọa độ giao thoa này, vì trong văn bản thực tế, nó chia sẻ chung các cấu trúc câu với cả "woman" và "king".

# 24. Application — Semantic Search

In [23]:
import gzip
import json
import nltk
import time
from google.colab import drive
from gensim.models import Word2Vec

nltk.download('punkt')

file_path = '/content/drive/MyDrive/Colab_Notebooks/c4-train.00000-of-01024-30K.json.gz'

# 3. Đọc dữ liệu từ file json.gz
print("\nĐang đọc dữ liệu từ file 30K...")
real_corpus = []
with gzip.open(file_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        if 'text' in data:
            real_corpus.append(data['text'])
print(f"Đã tải xong {len(real_corpus)} văn bản!")

start_time = time.time()
tokenized_real_corpus = [nltk.word_tokenize(doc.lower()) for doc in real_corpus]
print(f"Tokenize xong trong {time.time() - start_time:.2f} giây.")

real_model = Word2Vec(
    sentences=tokenized_real_corpus,
    vector_size=100,
    window=5,
    min_count=5,
    epochs=5
)
def semantic_search_30k(query, model, raw_docs, tokenized_docs, top_k=3):
    query_tokens = query.lower().split()

    expanded_query = set(query_tokens)
    for q_word in query_tokens:
        if q_word in model.wv:
            for sim_w, _ in model.wv.most_similar(q_word, topn=3):
                expanded_query.add(sim_w)

    print(f"\n** Query gốc: '{query}'")
    print(f"** Query mở rộng (Semantic): {expanded_query}\n")

    print("Đang chấm điểm 30,000 văn bản...")
    doc_scores = []

    for i, doc_tokens in enumerate(tokenized_docs):
        # Đếm số lượng từ khóa mở rộng có trong văn bản
        match_count = len(expanded_query.intersection(set(doc_tokens)))
        if match_count > 0:
            doc_scores.append((match_count, raw_docs[i]))

    # Sắp xếp giảm dần theo điểm số
    doc_scores.sort(key=lambda x: x[0], reverse=True)

    print("--- KẾT QUẢ TÌM KIẾM ---")
    for score, doc in doc_scores[:top_k]:
        snippet = doc[:300].replace('\n', ' ')
        print(f"Score: {score} | Snippet: {snippet}...\n")

# 7. Chạy thử nghiệm Semantic Search
semantic_search_30k(
    query="medical treatment",
    model=real_model,
    raw_docs=real_corpus,
    tokenized_docs=tokenized_real_corpus
)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!



Đang đọc dữ liệu từ file 30K...
Đã tải xong 30000 văn bản!
Tokenize xong trong 78.95 giây.

** Query gốc: 'medical treatment'
** Query mở rộng (Semantic): {'surgery', 'treatment', 'surgical', 'veterinary', 'medical', 'nursing', 'counseling', 'therapy'}

Đang chấm điểm 30,000 văn bản...
--- KẾT QUẢ TÌM KIẾM ---
Score: 6 | Snippet: Surgery affects people in a variety of ways, but as you begin to emerge from anesthesia, you’re likely to want to alert your nurse to any issues you may have. LASIK eye surgery is done in offices all over the country by experienced eye surgeons and you need to be in a position to discover a great su...

Score: 6 | Snippet: Pediatric pituitary adenomas are rare, accounting for <3 % of all childhood intracranial tumors, the majority of which are prolactinomas. Consequently, they are often misdiagnosed as other suprasellar masses such as craniopharyngiomas in this age group. Whilst guidelines exist for the treatment of a...

Score: 6 | Snippet: 1. What are some 

# 26. Một phần rất quan trọng — Polysemy

- Cơ chế nội suy trong không gian vector: Nếu trong corpus, từ "bank" xuất hiện 80% trong các bài báo tài chính (đi kèm ngữ cảnh money, deposit, account) và 20% trong các bài viết về tự nhiên (đi kèm river, water, sit), thuật toán sẽ kéo vector của "bank" nằm lơ lửng ở khoảng giữa hai cụm từ vựng này, nhưng lệch trọng tâm về phía cụm tài chính. Nó không đại diện sắc nét cho bất kỳ nghĩa nào, mà là sự pha trộn của tất cả.

- Static Embeddings: Việc gộp chung này tạo ra sự nhiễu loạn ngữ nghĩa. Khi ta xử lý câu "We sat on the river bank", Word2Vec vẫn dùng chính xác vector đó, mang theo 80% ý nghĩa "tiền bạc/tài chính" vào tính toán, dù nó hoàn toàn sai bối cảnh.